# 链式法则与反向传播：PyTorch逐步演示

目标：用一个单神经元例子，逐步展示
1. 前向计算（z, a, L）
2. 反向传播（dL/dw, dL/db）
3. 参数更新前后对比（w, b, L）

每一步都打印具体数值，避免“只看代码看不懂”。

In [ ]:
pip install torch --index-url https://download.pytorch.org/whl/cpu
import torch
torch.set_printoptions(precision=6, sci_mode=False)
print('torch version:', torch.__version__)

ModuleNotFoundError: No module named 'torch'

## 1) 准备数据与参数

In [ ]:
# 单样本数据
x = torch.tensor(2.0)
y = torch.tensor(1.0)

# 可训练参数（requires_grad=True 才能自动求导）
w = torch.tensor(0.5, requires_grad=True)
b = torch.tensor(0.1, requires_grad=True)

lr = 0.1

print('x =', x.item())
print('y =', y.item())
print('w(init) =', w.item())
print('b(init) =', b.item())
print('learning rate =', lr)

x = 2.0
y = 1.0
w(init) = 0.5
b(init) = 0.10000000149011612
learning rate = 0.1


## 2) 前向传播（逐步打印）

In [ ]:
# z = wx + b
z = w * x + b
print('[Forward] z = w*x + b =', z.item())

# a = sigmoid(z)
a = torch.sigmoid(z)
print('[Forward] a = sigmoid(z) =', a.item())

# L = 1/2 * (a-y)^2
L = 0.5 * (a - y) ** 2
print('[Forward] L = 0.5*(a-y)^2 =', L.item())

[Forward] z = w*x + b = 1.100000023841858
[Forward] a = sigmoid(z) = 0.7502601146697998
[Forward] L = 0.5*(a-y)^2 = 0.031185004860162735


## 3) 反向传播（自动求梯度）

In [ ]:
# 清空历史梯度（良好习惯）
if w.grad is not None:
    w.grad.zero_()
if b.grad is not None:
    b.grad.zero_()

# 计算梯度 dL/dw, dL/db
L.backward()

print('[Backward] dL/dw =', w.grad.item())
print('[Backward] dL/db =', b.grad.item())

[Backward] dL/dw = -0.09358745813369751
[Backward] dL/db = -0.046793729066848755


## 4) 参数更新（手动梯度下降）

In [ ]:
w_before = w.item()
b_before = b.item()
L_before = L.item()

with torch.no_grad():
    w -= lr * w.grad
    b -= lr * b.grad

print('[Update] w: {:.6f} -> {:.6f}'.format(w_before, w.item()))
print('[Update] b: {:.6f} -> {:.6f}'.format(b_before, b.item()))

[Update] w: 0.500000 -> 0.509359
[Update] b: 0.100000 -> 0.104679


## 5) 更新后再做一次前向，验证损失是否下降

In [ ]:
z_new = w * x + b
a_new = torch.sigmoid(z_new)
L_new = 0.5 * (a_new - y) ** 2

print('[Check] z_new =', z_new.item())
print('[Check] a_new =', a_new.item())
print('[Check] L_before = {:.6f}, L_after = {:.6f}'.format(L_before, L_new.item()))
print('[Check] loss decreased?', L_new.item() < L_before)

[Check] z_new = 1.123396873474121
[Check] a_new = 0.754618227481842
[Check] L_before = 0.031185, L_after = 0.030106
[Check] loss decreased? True


## 6) 一段最小训练循环（3轮）

In [ ]:
# 重新初始化，演示循环
w = torch.tensor(0.5, requires_grad=True)
b = torch.tensor(0.1, requires_grad=True)
lr = 0.1

for epoch in range(1, 4):
    # forward
    z = w * x + b
    a = torch.sigmoid(z)
    L = 0.5 * (a - y) ** 2

    # backward
    if w.grad is not None: w.grad.zero_()
    if b.grad is not None: b.grad.zero_()
    L.backward()

    # update
    with torch.no_grad():
        w -= lr * w.grad
        b -= lr * b.grad

    print(f'epoch={epoch} | L={L.item():.6f} | w={w.item():.6f} | b={b.item():.6f} | dw={w.grad.item():.6f} | db={b.grad.item():.6f}')

epoch=1 | L=0.031185 | w=0.509359 | b=0.104679 | dw=-0.093587 | db=-0.046794
epoch=2 | L=0.030106 | w=0.518446 | b=0.109223 | dw=-0.090874 | db=-0.045437
epoch=3 | L=0.029089 | w=0.527275 | b=0.113638 | dw=-0.088290 | db=-0.044145
